# Day 18 — Advanced Programming Concepts (Week 3 Capstone)
 
**Week 3: Functions & Functional Programming**
 
This is the integrative day for the week — combining functions, closures, lambdas, functional tools (`map`/`filter`/`reduce`), decorators, generators, file I/O, and exception handling into more realistic, multi-concept problems.

### Q1. Combine Decorator + Exception Handling
Write a decorator `safe_call` that wraps any function, catches **any** exception raised inside it, prints a friendly error message including the exception type, and returns `None` instead of crashing. Apply it to a function that divides two numbers.

In [1]:
def safe_call(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"Error occurred in function '{func.__name__}': {type(e).__name__} - {e}")
            return None
    return wrapper


@safe_call
def divide(a, b):
    return a / b


# Example usage
print(divide(10, 2))   # 5.0
print(divide(10, 0))   # handled safely, returns None
print(divide("10", 2)) # handled safely, returns None

5.0
Error occurred in function 'divide': ZeroDivisionError - division by zero
None
Error occurred in function 'divide': TypeError - unsupported operand type(s) for /: 'str' and 'int'
None


### Q2. Combine Generator + File Reading
Write a generator function `read_large_file(filename)` that yields one line at a time from a file (rather than reading it all into memory at once with `.readlines()`). Use it to process a file you create with at least 20 lines, printing only lines containing a specific keyword.
 

In [2]:
# Generator that reads file line by line
def read_large_file(filename):
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            yield line.strip()


# Create a sample file with at least 20 lines
filename = "sample.txt"

with open(filename, "w", encoding="utf-8") as f:
    for i in range(1, 26):  # 25 lines
        if i % 5 == 0:
            f.write(f"Line {i}: python is powerful\n")
        else:
            f.write(f"Line {i}: just some random text\n")


# Keyword to search
keyword = "python"

# Process file using generator
for line in read_large_file(filename):
    if keyword in line:
        print(line)

Line 5: python is powerful
Line 10: python is powerful
Line 15: python is powerful
Line 20: python is powerful
Line 25: python is powerful


### Q3. Combine Closures + Validation
Write a function `make_range_checker(low, high)` that returns a closure `check(value)` raising a custom exception `OutOfRangeError` if `value` falls outside `[low, high]`, otherwise returning `True`. Test with at least two different checkers and several values, including boundary cases.

In [3]:
# Custom Exception
class OutOfRangeError(Exception):
    pass


# Closure factory
def make_range_checker(low, high):
    def check(value):
        if value < low or value > high:
            raise OutOfRangeError(
                f"{value} is outside allowed range [{low}, {high}]"
            )
        return True

    return check


# Create two different checkers
age_checker = make_range_checker(18, 60)
temp_checker = make_range_checker(-10, 50)


# Test cases
tests = [
    ("age_checker", age_checker, [17, 18, 30, 60, 61]),
    ("temp_checker", temp_checker, [-10, 0, 25, 50, 51]),
]


for name, checker, values in tests:
    print(f"\nTesting {name}")
    for v in values:
        try:
            result = checker(v)
            print(f"{v}: {result}")
        except OutOfRangeError as e:
            print(f"{v}: ERROR -> {e}")


Testing age_checker
17: ERROR -> 17 is outside allowed range [18, 60]
18: True
30: True
60: True
61: ERROR -> 61 is outside allowed range [18, 60]

Testing temp_checker
-10: True
0: True
25: True
50: True
51: ERROR -> 51 is outside allowed range [-10, 50]


### Q4. Functional Pipeline with Exception-Safe Steps
Given a list of strings representing numbers, some of which are invalid (e.g., `["4", "7", "oops", "12", "x", "9"]`), write a pipeline using `filter()` (with a helper function, not necessarily lambda) to discard invalid entries, then `map()` to convert valid ones to integers, then `functools.reduce()` to sum them — all without the program crashing on the invalid entries.

In [4]:
from functools import reduce


# Helper function: safely check if a string is an integer
def is_valid_int(s):
    try:
        int(s)
        return True
    except ValueError:
        return False


# Input data
data = ["4", "7", "oops", "12", "x", "9"]


# Step 1: filter valid numeric strings
valid_strings = filter(is_valid_int, data)


# Step 2: convert to integers
numbers = map(int, valid_strings)


# Step 3: reduce to sum
total = reduce(lambda a, b: a + b, numbers, 0)


print("Sum:", total)

Sum: 32


### Q5. Decorator for Retrying File Operations
Combine ideas from decorators and file I/O: write a decorator `retry_on_failure(max_attempts)` that retries a wrapped function up to `max_attempts` times if it raises an `IOError`/`OSError`, with a short message printed on each failed attempt, and re-raises after the final attempt fails.

In [5]:
import time
from functools import wraps


def retry_on_failure(max_attempts):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None

            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except (IOError, OSError) as e:
                    last_exception = e
                    print(f"Attempt {attempt} failed: {type(e).__name__} - {e}")

                    # small delay for realism (optional)
                    time.sleep(0.5)

            # After all retries fail
            print("All retry attempts exhausted.")
            raise last_exception

        return wrapper
    return decorator

In [6]:
@retry_on_failure(max_attempts=3)
def read_file(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return f.read()

In [7]:
try:
    content = read_file("missing_file.txt")
    print(content)
except Exception as e:
    print("Final error:", e)

Attempt 1 failed: FileNotFoundError - [Errno 2] No such file or directory: 'missing_file.txt'
Attempt 2 failed: FileNotFoundError - [Errno 2] No such file or directory: 'missing_file.txt'
Attempt 3 failed: FileNotFoundError - [Errno 2] No such file or directory: 'missing_file.txt'
All retry attempts exhausted.
Final error: [Errno 2] No such file or directory: 'missing_file.txt'


### Q6. Generator-Based CSV Row Processor with Error Skipping
Write a generator function `process_csv_rows(filename)` that reads a CSV file row by row (yielding one processed row at a time), validates that a designated numeric column can be converted to `float`, and skips (with a printed warning) any row where conversion fails, without stopping the whole generator.

In [8]:
import csv


def process_csv_rows(filename, numeric_column="value"):
    """
    Generator that reads CSV row by row,
    validates numeric column, and skips bad rows safely.
    """

    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for line_num, row in enumerate(reader, start=2):  # header is line 1
            try:
                # Validate numeric column
                row[numeric_column] = float(row[numeric_column])
                yield row

            except (ValueError, TypeError, KeyError) as e:
                print(
                    f"Warning (line {line_num}): "
                    f"Skipping row due to invalid '{numeric_column}' -> {row.get(numeric_column)}"
                )
                continue

In [9]:
filename = "data.csv"

with open(filename, "w", encoding="utf-8") as f:
    f.write("name,value\n")
    f.write("A,10\n")
    f.write("B,20.5\n")
    f.write("C,not_a_number\n")
    f.write("D,40\n")
    f.write("E,?\n")
    f.write("F,60\n")

In [10]:
for row in process_csv_rows("data.csv", numeric_column="value"):
    print("Processed:", row)

Processed: {'name': 'A', 'value': 10.0}
Processed: {'name': 'B', 'value': 20.5}
Warning (line 4): Skipping row due to invalid 'value' -> not_a_number
Processed: {'name': 'D', 'value': 40.0}
Warning (line 6): Skipping row due to invalid 'value' -> ?
Processed: {'name': 'F', 'value': 60.0}


### Q7. Memoized Recursive Function via Custom Decorator
Write your own `@memoize` decorator (similar to Day 16, but applied here in a fresh combined context) and apply it to a recursive function that computes the number of ways to climb `n` stairs taking 1 or 2 steps at a time (a Fibonacci-like recurrence). Test for `n = 30` and explain why memoization is necessary at that scale.

In [11]:
from functools import wraps

def memoize(func):
    cache = {}

    @wraps(func)
    def wrapper(n):
        if n in cache:
            return cache[n]

        result = func(n)
        cache[n] = result
        return result

    return wrapper

In [12]:
@memoize
def climb_ways(n):
    # Base cases
    if n == 0 or n == 1:
        return 1

    return climb_ways(n - 1) + climb_ways(n - 2)

In [13]:
import time

start = time.time()
result = climb_ways(30)
end = time.time()

print("Ways to climb 30 stairs:", result)
print("Execution time:", end - start, "seconds")

Ways to climb 30 stairs: 1346269
Execution time: 6.079673767089844e-05 seconds


### Q8. Logging Decorator that Writes to a File
Write a decorator `file_logger` that, every time the wrapped function is called, appends a line to a log file (`function_log.txt`) recording the function name and the arguments it was called with. Apply it to two or three different functions and inspect the resulting log file's contents afterward.

In [14]:
from functools import wraps

def file_logger(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        log_line = f"Function: {func.__name__}, args: {args}, kwargs: {kwargs}\n"

        with open("function_log.txt", "a", encoding="utf-8") as log_file:
            log_file.write(log_line)

        return func(*args, **kwargs)

    return wrapper

In [15]:
@file_logger
def add(a, b):
    return a + b


@file_logger
def multiply(a, b):
    return a * b


@file_logger
def greet(name, message="Hello"):
    return f"{message}, {name}!"

In [16]:
print(add(3, 5))
print(multiply(4, 6))
print(greet("Alice"))
print(greet("Bob", message="Hi"))

8
24
Hello, Alice!
Hi, Bob!


### Q9. Closures for Configurable Validators Used in a Filter Pipeline
Write a closure-based factory `make_length_validator(min_len)` returning a function that checks if a string meets a minimum length. Use the resulting validator function inside `filter()` to clean a list of usernames, removing any that are too short.

In [17]:
def make_length_validator(min_len):
    def validator(s):
        return len(s) >= min_len
    return validator

In [18]:
usernames = [
    "alex",
    "bo",
    "charlie",
    "x",
    "maria",
    "jo",
    "sophia",
    "li"
]

In [19]:
# Minimum length requirement
min_length = 3

validator = make_length_validator(min_length)

# Filter invalid usernames
clean_usernames = list(filter(validator, usernames))

print("Valid usernames:", clean_usernames)

Valid usernames: ['alex', 'charlie', 'maria', 'sophia']


### Q10. Custom Exception Hierarchy for a Mini "Banking System"
Design a small exception hierarchy: `BankError(Exception)`, with subclasses `InsufficientFundsError` and `InvalidAmountError`. Write functions `deposit(balance, amount)` and `withdraw(balance, amount)` that raise the appropriate custom exceptions for invalid (e.g., negative) amounts or insufficient funds. Write a small driver section that performs several operations, catching and printing meaningful messages for each type of error.

In [25]:
class BankError(Exception):
    """Base class for all banking-related errors"""
    pass


class InsufficientFundsError(BankError):
    """Raised when withdrawal exceeds balance"""
    pass


class InvalidAmountError(BankError):
    """Raised when amount is zero or negative"""
    pass

In [26]:
def deposit(balance, amount):
    if amount <= 0:
        raise InvalidAmountError(f"Deposit failed: invalid amount {amount}")

    balance += amount
    return balance

In [27]:
def withdraw(balance, amount):
    if amount <= 0:
        raise InvalidAmountError(f"Withdrawal failed: invalid amount {amount}")

    if amount > balance:
        raise InsufficientFundsError(
            f"Withdrawal failed: requested {amount}, available {balance}"
        )

    balance -= amount
    return balance

In [28]:
balance = 1000

operations = [
    ("deposit", 500),
    ("withdraw", 200),
    ("withdraw", 2000),   # insufficient funds
    ("deposit", -50),     # invalid amount
    ("withdraw", -10),    # invalid amount
]

for op, amount in operations:
    try:
        if op == "deposit":
            balance = deposit(balance, amount)
            print(f"Deposited {amount}. New balance: {balance}")

        elif op == "withdraw":
            balance = withdraw(balance, amount)
            print(f"Withdrew {amount}. New balance: {balance}")

    except InsufficientFundsError as e:
        print(f"[INSUFFICIENT FUNDS] {e}")

    except InvalidAmountError as e:
        print(f"[INVALID AMOUNT] {e}")

    except BankError as e:
        print(f"[BANK ERROR] {e}")

Deposited 500. New balance: 1500
Withdrew 200. New balance: 1300
[INSUFFICIENT FUNDS] Withdrawal failed: requested 2000, available 1300
[INVALID AMOUNT] Deposit failed: invalid amount -50
[INVALID AMOUNT] Withdrawal failed: invalid amount -10


### Q11. Generator Pipeline Reading from a Simulated Data Stream with Exception Safety
Create a list of 15 dictionaries simulating sensor readings, where a few intentionally have a missing or invalid `"value"` key. Write a generator function `clean_readings(data)` that yields only the readings with a valid numeric `"value"`, using `try`/`except` internally to skip bad entries silently (but counting how many were skipped, reported at the end).

In [29]:
data = [
    {"id": 1, "value": 10},
    {"id": 2, "value": 12.5},
    {"id": 3, "value": "NaN"},     # invalid
    {"id": 4, "value": 15},
    {"id": 5},                     # missing
    {"id": 6, "value": 18},
    {"id": 7, "value": None},     # invalid
    {"id": 8, "value": 21},
    {"id": 9, "value": "error"},  # invalid
    {"id": 10, "value": 25},
    {"id": 11, "value": 30},
    {"id": 12},                   # missing
    {"id": 13, "value": 35.5},
    {"id": 14, "value": 40},
    {"id": 15, "value": "x"}      # invalid
]

In [30]:
def clean_readings(data):
    skipped = 0

    for reading in data:
        try:
            value = float(reading["value"])  # may raise KeyError or ValueError
            yield {**reading, "value": value}

        except (KeyError, TypeError, ValueError):
            skipped += 1
            continue

    print(f"\nTotal skipped readings: {skipped}")

In [31]:
cleaned = list(clean_readings(data))

print("Valid readings:")
for r in cleaned:
    print(r)


Total skipped readings: 5
Valid readings:
{'id': 1, 'value': 10.0}
{'id': 2, 'value': 12.5}
{'id': 3, 'value': nan}
{'id': 4, 'value': 15.0}
{'id': 6, 'value': 18.0}
{'id': 8, 'value': 21.0}
{'id': 10, 'value': 25.0}
{'id': 11, 'value': 30.0}
{'id': 13, 'value': 35.5}
{'id': 14, 'value': 40.0}


### Q12. Decorator that Converts Function Output Using a Provided Function
Write a decorator factory `transform_output(transformer)` that takes a function `transformer` (e.g., a lambda) and returns a decorator which applies `transformer` to whatever the wrapped function returns. Demonstrate it by wrapping a function that returns a list of numbers, with a transformer that sorts the list.

In [32]:
from functools import wraps

def transform_output(transformer):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            return transformer(result)
        return wrapper
    return decorator

In [33]:
sort_transformer = lambda x: sorted(x)

In [34]:
@transform_output(sort_transformer)
def get_numbers():
    return [42, 3, 17, 8, 99, 1]

In [35]:
print(get_numbers())

[1, 3, 8, 17, 42, 99]


### Q13. JSON-Backed Configuration Loader with Defaults and Validation
Write a function `load_config(filename, required_keys)` that:
- Tries to load a JSON config file
- Raises a custom `ConfigError` if the file is missing or invalid JSON
- Raises a custom `ConfigError` if any of `required_keys` are missing from the loaded dictionary
- Returns the validated config dictionary otherwise
Test it against a valid config file and at least one invalid scenario.
 

In [36]:
class ConfigError(Exception):
    """Raised for any configuration loading or validation failure"""
    pass

In [37]:
import json

def load_config(filename, required_keys):
    try:
        with open(filename, "r", encoding="utf-8") as f:
            config = json.load(f)

    except FileNotFoundError:
        raise ConfigError(f"Config file '{filename}' not found")

    except json.JSONDecodeError as e:
        raise ConfigError(f"Invalid JSON in config file: {e}")

    # Validate required keys
    missing_keys = [key for key in required_keys if key not in config]

    if missing_keys:
        raise ConfigError(f"Missing required config keys: {missing_keys}")

    return config

In [38]:
valid_file = "config_valid.json"

with open(valid_file, "w", encoding="utf-8") as f:
    json.dump({
        "host": "localhost",
        "port": 8080,
        "debug": True
    }, f)

In [39]:
invalid_file = "config_invalid.json"

with open(invalid_file, "w", encoding="utf-8") as f:
    f.write("{ host: localhost, port: }")  # malformed JSON

In [40]:
missing_file = "config_missing.json"

with open(missing_file, "w", encoding="utf-8") as f:
    json.dump({
        "host": "localhost"
    }, f)

In [41]:
required = ["host", "port", "debug"]

files = [valid_file, invalid_file, missing_file, "nonexistent.json"]

for file in files:
    print(f"\nLoading: {file}")
    try:
        config = load_config(file, required)
        print("Loaded config:", config)

    except ConfigError as e:
        print("ConfigError:", e)


Loading: config_valid.json
Loaded config: {'host': 'localhost', 'port': 8080, 'debug': True}

Loading: config_invalid.json
ConfigError: Invalid JSON in config file: Expecting property name enclosed in double quotes: line 1 column 3 (char 2)

Loading: config_missing.json
ConfigError: Missing required config keys: ['port', 'debug']

Loading: nonexistent.json
ConfigError: Config file 'nonexistent.json' not found


### Q14. Recursive Generator for Directory-Like Nested Structure
Given a nested dictionary structure representing a fake file system (folders containing files and other folders), write a recursive generator function `walk(structure, path="")` that yields the full path of every file found, similar to how `os.walk` traverses real directories.
 

In [42]:
filesystem = {
    "home": {
        "user": {
            "docs": {
                "a.txt": None,
                "b.txt": None,
            },
            "pics": {
                "img1.png": None,
                "img2.png": None,
            }
        }
    },
    "etc": {
        "config.json": None
    },
    "readme.md": None
}

In [43]:
def walk(structure, path=""):
    for name, content in structure.items():
        current_path = f"{path}/{name}" if path else name

        # If it's a folder (dict), recurse
        if isinstance(content, dict):
            yield from walk(content, current_path)

        # Otherwise it's a file
        else:
            yield current_path

In [44]:
for file_path in walk(filesystem):
    print(file_path)

home/user/docs/a.txt
home/user/docs/b.txt
home/user/pics/img1.png
home/user/pics/img2.png
etc/config.json
readme.md


### Q15. Functional + OOP-lite: Strategy Selection via Dictionary of Functions
Write a dictionary `strategies = {"add": ..., "sub": ..., "mul": ...}` mapping names to lambda or regular functions. Write a function `execute_strategy(name, a, b)` that looks up and calls the correct function, raising a custom `UnknownStrategyError` if `name` isn't in the dictionary.
 

In [45]:
class UnknownStrategyError(Exception):
    """Raised when an invalid strategy name is requested"""
    pass

In [46]:
strategies = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "div": lambda a, b: a / b if b != 0 else float("inf")
}

In [47]:
def execute_strategy(name, a, b):
    try:
        func = strategies[name]
    except KeyError:
        raise UnknownStrategyError(f"Strategy '{name}' is not supported")

    return func(a, b)

In [48]:
tests = [
    ("add", 10, 5),
    ("sub", 10, 5),
    ("mul", 10, 5),
    ("div", 10, 5),
    ("mod", 10, 5)  # invalid strategy
]

for name, a, b in tests:
    try:
        result = execute_strategy(name, a, b)
        print(f"{name}({a}, {b}) = {result}")

    except UnknownStrategyError as e:
        print("ERROR:", e)

add(10, 5) = 15
sub(10, 5) = 5
mul(10, 5) = 50
div(10, 5) = 2.0
ERROR: Strategy 'mod' is not supported


### Q16. Decorator Stack: Timing + Logging + Exception Handling Together
Apply three decorators together (`@timer`, `@file_logger` or similar, and `@safe_call` from Q1) to a single function that performs a moderately expensive computation and sometimes raises an exception (simulate randomness). Observe and explain (in a comment) the order in which the decorators' effects appear.

In [49]:
def safe_call(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"[SAFE_CALL] Caught {type(e).__name__}: {e}")
            return None
    return wrapper

In [50]:
def file_logger(func):
    def wrapper(*args, **kwargs):
        with open("function_log.txt", "a", encoding="utf-8") as f:
            f.write(f"Called {func.__name__} with args={args}, kwargs={kwargs}\n")
        return func(*args, **kwargs)
    return wrapper

In [51]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"[TIMER] {func.__name__} took {end - start:.6f}s")
        return result
    return wrapper

In [52]:
import random
import time

@safe_call
@file_logger
@timer
def risky_computation(n):
    """Simulates computation that may fail randomly"""

    time.sleep(0.2)  # simulate workload

    if random.random() < 0.4:
        raise ValueError("Random failure occurred!")

    return sum(i * i for i in range(n))

In [53]:
for i in range(5):
    print("\nRun:", i + 1)
    result = risky_computation(1000)
    print("Result:", result)


Run: 1
[SAFE_CALL] Caught ValueError: Random failure occurred!
Result: None

Run: 2
[TIMER] risky_computation took 0.200245s
Result: 332833500

Run: 3
[TIMER] risky_computation took 0.200613s
Result: 332833500

Run: 4
[SAFE_CALL] Caught ValueError: Random failure occurred!
Result: None

Run: 5
[SAFE_CALL] Caught ValueError: Random failure occurred!
Result: None


### Q17. Generator-Based Batch Processor with Resumability Simulation
Write a generator function `batch_processor(items, batch_size)` that yields batches (lists) of items, and simulate "resuming" processing by manually calling `next()` a few times, stopping partway, and then continuing the loop afterward — printing which batch number is being processed each time.

In [54]:
def batch_processor(items, batch_size):
    batch = []
    batch_num = 1

    for item in items:
        batch.append(item)

        if len(batch) == batch_size:
            print(f"Yielding batch {batch_num}")
            yield batch
            batch = []
            batch_num += 1

    # yield remaining items
    if batch:
        print(f"Yielding batch {batch_num}")
        yield batch

In [55]:
items = list(range(1, 13))  # 1 to 12

In [56]:
gen = batch_processor(items, 3)

In [57]:
print("First call:", next(gen))   # batch 1
print("Second call:", next(gen))  # batch 2

Yielding batch 1
First call: [1, 2, 3]
Yielding batch 2
Second call: [4, 5, 6]


In [58]:
print("\nResuming loop...\n")

for batch in gen:
    print("Processed:", batch)


Resuming loop...

Yielding batch 3
Processed: [7, 8, 9]
Yielding batch 4
Processed: [10, 11, 12]


### Q18. Custom Context Manager Function-Based (via `contextlib`)
Using `@contextlib.contextmanager`, write a custom context manager function `temporary_file(filename, content)` that creates a file with given content on entry and deletes it on exit (even if an exception occurs inside the `with` block). Demonstrate it being used safely.

In [59]:
import os
import contextlib

@contextlib.contextmanager
def temporary_file(filename, content):
    try:
        # Setup: create file
        with open(filename, "w", encoding="utf-8") as f:
            f.write(content)

        print(f"[ENTER] Created temporary file: {filename}")

        # Yield control to with-block
        yield filename

    finally:
        # Cleanup: always executed
        if os.path.exists(filename):
            os.remove(filename)
            print(f"[EXIT] Deleted temporary file: {filename}")

In [60]:
with temporary_file("temp.txt", "Hello World!") as fname:
    with open(fname, "r") as f:
        print("File content:", f.read())

print("Outside context")

[ENTER] Created temporary file: temp.txt
File content: Hello World!
[EXIT] Deleted temporary file: temp.txt
Outside context


In [61]:
try:
    with temporary_file("temp_fail.txt", "Important data") as fname:
        print("Reading file:", fname)

        # Simulate error
        raise ValueError("Something went wrong inside context")

except ValueError as e:
    print("Caught exception:", e)

print("Program continues safely")

[ENTER] Created temporary file: temp_fail.txt
Reading file: temp_fail.txt
[EXIT] Deleted temporary file: temp_fail.txt
Caught exception: Something went wrong inside context
Program continues safely


### Q19. Full Pipeline — Read, Clean, Transform, Aggregate
Create a small CSV file with columns `name,score` and at least 10 rows, including 2–3 rows with intentionally invalid (non-numeric) scores. Write a complete pipeline that: (1) reads the CSV using a generator, (2) filters out rows with invalid scores (using `try`/`except` inside a helper function used by `filter`), (3) maps valid rows to just their numeric scores, and (4) reduces them to compute both the total and the average score.

In [67]:
filename = "scores.csv"

with open(filename, "w", encoding="utf-8") as f:
    f.write("name,score\n")
    f.write("A,10\n")
    f.write("B,15\n")
    f.write("C,NaN\n")
    f.write("D,20\n")
    f.write("E,error\n")
    f.write("F,25\n")
    f.write("G,30\n")
    f.write("H,?\n")
    f.write("I,35\n")
    f.write("J,40\n")

In [68]:
import csv

def read_csv_rows(filename):
    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            yield row

In [69]:
def is_valid_score(row):
    try:
        float(row["score"])
        return True
    except (ValueError, TypeError, KeyError):
        return False

In [70]:
from functools import reduce

rows = read_csv_rows(filename)

# Step 1: filter valid rows
valid_rows = filter(is_valid_score, rows)

# Step 2: extract numeric scores
scores = map(lambda r: float(r["score"]), valid_rows)

# Step 3: convert to list (needed for avg calculation)
scores_list = list(scores)

# Step 4: aggregation
total = reduce(lambda a, b: a + b, scores_list, 0)
average = total / len(scores_list) if scores_list else 0

In [71]:
print("Total Score:", total)
print("Average Score:", average)

Total Score: nan
Average Score: nan


### Q20. Custom Exception with a `__str__` Override
Define a custom exception `ValidationError` that overrides `__str__` to produce a nicely formatted message including a field name and the invalid value that caused the error (stored as attributes set in `__init__`). Raise and catch it, printing the exception directly to show the custom formatting in action.

In [72]:
class ValidationError(Exception):
    def __init__(self, field, value, message="Invalid value provided"):
        self.field = field
        self.value = value
        self.message = message
        super().__init__(self.message)

    def __str__(self):
        return (
            f"ValidationError:\n"
            f"  Field : {self.field}\n"
            f"  Value : {self.value}\n"
            f"  Error : {self.message}"
        )

In [73]:
def validate_age(age):
    if not isinstance(age, int) or age < 0 or age > 120:
        raise ValidationError(
            field="age",
            value=age,
            message="Age must be an integer between 0 and 120"
        )
    return True

In [74]:
test_values = [25, -5, "abc", 130, 40]

for value in test_values:
    try:
        print(f"\nChecking value: {value}")
        validate_age(value)
        print("Valid age ✔")

    except ValidationError as e:
        print(e)   # triggers __str__ formatting


Checking value: 25
Valid age ✔

Checking value: -5
ValidationError:
  Field : age
  Value : -5
  Error : Age must be an integer between 0 and 120

Checking value: abc
ValidationError:
  Field : age
  Value : abc
  Error : Age must be an integer between 0 and 120

Checking value: 130
ValidationError:
  Field : age
  Value : 130
  Error : Age must be an integer between 0 and 120

Checking value: 40
Valid age ✔


### Q21. Decorator for Caching Results to a File (Persistent Memoization)
Write a decorator `disk_cache(filename)` that, the first time a wrapped function is called with a particular argument, computes and stores the result in a JSON file; on subsequent runs (simulate by calling twice within the same script), it should be able to retrieve previously cached results from the file instead of recomputing. (Keys can be simplified to support only single, JSON-serializable arguments.)

In [75]:
import json
import os
from functools import wraps

def disk_cache(filename):
    def decorator(func):
        @wraps(func)
        def wrapper(arg):
            # Load existing cache
            if os.path.exists(filename):
                with open(filename, "r", encoding="utf-8") as f:
                    try:
                        cache = json.load(f)
                    except json.JSONDecodeError:
                        cache = {}
            else:
                cache = {}

            key = str(arg)  # simplified single-arg key

            # Cache hit
            if key in cache:
                print(f"[CACHE HIT] {func.__name__}({arg})")
                return cache[key]

            # Cache miss → compute
            print(f"[CACHE MISS] Computing {func.__name__}({arg})")
            result = func(arg)

            cache[key] = result

            # Save back to disk
            with open(filename, "w", encoding="utf-8") as f:
                json.dump(cache, f)

            return result

        return wrapper
    return decorator

In [76]:
import time

@disk_cache("cache.json")
def slow_square(x):
    time.sleep(1)  # simulate expensive work
    return x * x

In [77]:
print("First run:")
print(slow_square(4))
print(slow_square(5))

First run:
[CACHE MISS] Computing slow_square(4)
16
[CACHE MISS] Computing slow_square(5)
25


In [78]:
print("\nSecond run (same calls):")
print(slow_square(4))
print(slow_square(5))


Second run (same calls):
[CACHE HIT] slow_square(4)
16
[CACHE HIT] slow_square(5)
25


### Q22. Recursive Function + Custom Exception for Depth Limiting
Write a recursive function `deep_sum(nested_list, max_depth=5, current_depth=0)` that sums all numbers in an arbitrarily nested list, but raises a custom `MaxDepthExceededError` if the nesting exceeds `max_depth`, to prevent runaway recursion on malformed data.

In [79]:
class MaxDepthExceededError(Exception):
    """Raised when recursion exceeds allowed nesting depth"""
    pass

In [80]:
def deep_sum(nested_list, max_depth=5, current_depth=0):

    if current_depth > max_depth:
        raise MaxDepthExceededError(
            f"Maximum depth exceeded: {current_depth} > {max_depth}"
        )

    total = 0

    for item in nested_list:
        if isinstance(item, list):
            total += deep_sum(
                item,
                max_depth=max_depth,
                current_depth=current_depth + 1
            )
        else:
            # assume numeric
            total += item

    return total

In [81]:
data_valid = [1, [2, 3], [4, [5, 6]], 7]

In [82]:
data_deep = [1, [2, [3, [4, [5, [6, [7]]]]]]]

In [83]:
try:
    result = deep_sum(data_valid, max_depth=5)
    print("Valid sum:", result)
except MaxDepthExceededError as e:
    print("Error:", e)

Valid sum: 28


In [84]:
try:
    result = deep_sum(data_deep, max_depth=3)
    print("Deep sum:", result)
except MaxDepthExceededError as e:
    print("Error:", e)

Error: Maximum depth exceeded: 4 > 3


### Q23. Generator Expression Combined with `functools.reduce` for One-Liner Aggregation
Given a list of dictionaries representing transactions (`[{"type": "credit", "amount": 100}, {"type": "debit", "amount": 40}, ...]`), use a generator expression together with `reduce()` to compute the net balance (sum of credits minus sum of debits) in a single expression (or a very small number of lines).

In [86]:
transactions = [
    {"type": "credit", "amount": 100},
    {"type": "debit", "amount": 40},
    {"type": "credit", "amount": 200},
    {"type": "debit", "amount": 50},
    {"type": "credit", "amount": 25},
]

In [87]:
from functools import reduce

net = reduce(
    lambda acc, x: acc + x,
    (
        t["amount"] if t["type"] == "credit" else -t["amount"]
        for t in transactions
    ),
    0
)

print("Net Balance:", net)

Net Balance: 235


### Q24. Building a Mini "Event System" with Closures and Decorators
Write a simple event system: a function `make_event_system()` that returns two functions, `on(event_name, handler)` (registers a handler function for an event) and `emit(event_name, *args)` (calls all handlers registered for that event with the given arguments), using a closure over a dictionary to store handlers. Bonus: write a decorator `@catches_errors` to wrap handlers so one failing handler doesn't prevent others from running when an event is emitted.

In [91]:
def catches_errors(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"[HANDLER ERROR] {func.__name__}: {e}")
    return wrapper

In [92]:
def make_event_system():
    handlers = {}

    def on(event_name, handler):
        """Register a handler for an event"""
        handlers.setdefault(event_name, []).append(handler)

    def emit(event_name, *args):
        """Trigger all handlers for an event"""
        if event_name not in handlers:
            print(f"[EVENT] No handlers for '{event_name}'")
            return

        print(f"[EVENT] Emitting '{event_name}' with args {args}")

        for handler in handlers[event_name]:
            handler(*args)

    return on, emit

In [93]:
on, emit = make_event_system()

In [94]:
@catches_errors
def greet_user(name):
    print(f"Hello, {name}!")

@catches_errors
def risky_handler(name):
    print(f"Processing {name}")
    raise ValueError("Something went wrong!")

@catches_errors
def farewell_user(name):
    print(f"Goodbye, {name}!")

In [95]:
on("user_join", greet_user)
on("user_join", risky_handler)
on("user_join", farewell_user)

In [96]:
emit("user_join", "Alice")

[EVENT] Emitting 'user_join' with args ('Alice',)
Hello, Alice!
Processing Alice
[HANDLER ERROR] risky_handler: Something went wrong!
Goodbye, Alice!


### Q25. Capstone Challenge — Resilient Data Import Tool
Build a more complete mini-tool that combines everything from this week:
1. Write a function that generates a sample JSON file containing a list of at least 12 "employee" records, each with `name`, `department`, and `salary` — but intentionally corrupt 2–3 records (missing field, wrong type, or negative salary).
2. Define a custom exception hierarchy (`ImportError`-style, but your own names to avoid shadowing the built-in) for `MissingFieldError` and `InvalidValueError`.
3. Write a generator function that reads records one at a time and validates each one, raising/catching the appropriate custom exception per record and skipping invalid ones while logging a summary of what was skipped and why.
4. Use functional tools (`map`/`filter`/`reduce` and/or comprehensions) to compute the average salary per department from the valid records only.
5. Wrap the whole import process in a decorator that times the operation and reports how many records were processed successfully vs skipped.
Print a final summary report of the results.

In [97]:
import json

def generate_employee_file(filename):
    data = [
        {"name": "Alice", "department": "IT", "salary": 70000},
        {"name": "Bob", "department": "HR", "salary": 50000},
        {"name": "Charlie", "department": "IT", "salary": "invalid"},   # bad
        {"name": "David", "department": "Finance", "salary": 90000},
        {"name": "Eva", "department": "HR", "salary": -30000},          # bad
        {"name": "Frank", "department": "IT", "salary": 80000},
        {"name": "Grace", "department": "Finance"},                     # missing
        {"name": "Hank", "department": "IT", "salary": 60000},
        {"name": "Ivy", "department": "HR", "salary": 55000},
        {"name": "Jack", "department": "Finance", "salary": 100000},
        {"name": "Kira", "department": "IT", "salary": 72000},
        {"name": "Leo", "department": "HR", "salary": 51000}
    ]

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    return filename

In [98]:
class BaseImportError(Exception):
    pass


class MissingFieldError(BaseImportError):
    pass


class InvalidValueError(BaseImportError):
    pass

In [99]:
def import_records(filename):
    import json

    skipped = []
    valid = []

    with open(filename, "r", encoding="utf-8") as f:
        records = json.load(f)

    for idx, rec in enumerate(records, start=1):
        try:
            # Validate required fields
            for field in ["name", "department", "salary"]:
                if field not in rec:
                    raise MissingFieldError(f"Row {idx}: missing {field}")

            salary = rec["salary"]

            # Validate type + value
            if not isinstance(salary, (int, float)):
                raise InvalidValueError(f"Row {idx}: salary not numeric")

            if salary < 0:
                raise InvalidValueError(f"Row {idx}: negative salary")

            valid.append(rec)
            yield rec

        except BaseImportError as e:
            print("[SKIPPED]", e)
            skipped.append(str(e))
            continue

    # attach summary to generator object (hacky but illustrative)
    import_records.skipped = skipped
    import_records.valid_count = len(valid)
    import_records.total_count = len(records)
    import_records.valid_records = valid

In [100]:
from functools import reduce
from collections import defaultdict

def compute_avg_salary(records):
    dept_map = defaultdict(list)

    # map step
    list(map(lambda r: dept_map[r["department"]].append(r["salary"]), records))

    # reduce-like aggregation
    return {
        dept: sum(salaries) / len(salaries)
        for dept, salaries in dept_map.items()
    }

In [101]:
import time
from functools import wraps

def timed_import(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()

        result = list(func(*args, **kwargs))  # consume generator

        end = time.time()

        total = getattr(func, "total_count", len(result))
        skipped = len(getattr(func, "skipped", []))
        valid = len(result)

        print("\n===== IMPORT SUMMARY =====")
        print(f"Total Records   : {total}")
        print(f"Valid Records   : {valid}")
        print(f"Skipped Records : {skipped}")
        print(f"Time Taken      : {end - start:.6f}s")

        return result

    return wrapper

In [102]:
@timed_import
def run_import(filename):
    return import_records(filename)

In [103]:
file = generate_employee_file("employees.json")

valid_records = run_import(file)

print("\n===== AVERAGE SALARY PER DEPARTMENT =====")
avg = compute_avg_salary(valid_records)

for dept, val in avg.items():
    print(dept, ":", round(val, 2))

[SKIPPED] Row 3: salary not numeric
[SKIPPED] Row 5: negative salary
[SKIPPED] Row 7: missing salary

===== IMPORT SUMMARY =====
Total Records   : 9
Valid Records   : 9
Skipped Records : 0
Time Taken      : 0.000237s

===== AVERAGE SALARY PER DEPARTMENT =====
IT : 70500.0
HR : 52000.0
Finance : 95000.0
